<a href="https://colab.research.google.com/github/basyalbhawana77/Forest_fire_analysis_Nepal/blob/main/1_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 1_data_clean.ipynb
# Project: stream_temp_project
# Purpose: Reshape and combine the raw stream temperature and station
# files into one clean, analysis-ready file.
# Reads: raw_data/modis_2015-2025.csv
# raw_data/Nepal_climate_data.csv
# Writes: clean_data/forest_fire_Nepal.csv
# =====================================================================
setwd("/content/drive/MyDrive/Forest_fire_analysis_Nepal")
library(readr) # read_csv(), write_csv()
library(dplyr) # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr) # pivot_longer()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [6]:
# 1. Read all 11 yearly MODIS CSVs from Modis_2015-2025
modis_files <- list.files("raw_data/Modis_2015-2025", pattern = "\\.csv$", full.names = TRUE)
modis_raw <- read_csv(modis_files, show_col_types = FALSE)

# 2. Read your NASA POWER climate CSVs from 'Nepal_climate_data_ 2015-2025'
climate_files <- list.files("raw_data/Nepal_climate_data_2015-2025", pattern = "\\.csv$", full.names = TRUE)

climate_raw <- lapply(climate_files, function(f){
  df <- read_csv(f, show_col_types = FALSE)
  names(df)[1] <- "parameter"
  return(df)
}) %>%
  bind_rows()


# Check raw row counts and columns
nrow(modis_raw)
nrow(climate_raw)
head(climate_raw)


Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or mo

[1] 42501

[1] 5616

parameter
<chr>
NASA/POWER Source Native Resolution Monthly and Annual
Dates (month/day/year): 01/01/2015 through 12/31/2025 in LST
Location: Regional
Elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = na meters
The value for missing source data that cannot be computed or is outside of the sources availability range: -999
Parameter(s):


In [12]:
# 1. Read all 11 yearly MODIS CSVs from raw_data/Modis_2015-2025/
modis_files <- list.files("raw_data/Modis_2015-2025", pattern = "\\.csv$", full.names = TRUE)
modis_raw <- read_csv(modis_files, show_col_types = FALSE)

# 2. Find the climate subfolder inside raw_data automatically and read all 8 CSVs
climate_dir <- list.dirs("raw_data", full.names = TRUE, recursive = FALSE)
climate_dir <- climate_dir[grepl("climate", climate_dir, ignore.case = TRUE)]
climate_files <- list.files(climate_dir, pattern = "\\.csv$", full.names = TRUE)
print(climate_files) # Verify all 8 climate files appear here

climate_raw <- read_csv(climate_files, skip = 9, show_col_types = FALSE)

# 3. Clean MODIS active fire detections
modis_clean <- modis_raw %>%
  rename_with(tolower) %>%
  filter(type == 0) %>%                                                    # Keep only vegetation/forest fires
  filter(confidence >= 30, !is.na(latitude), !is.na(longitude)) %>%        # Drop low-confidence & missing coordinates
  distinct(latitude, longitude, acq_date, acq_time, .keep_all = TRUE) %>%  # Remove duplicate detections
  mutate(
    acq_date = as.Date(acq_date),
    year     = as.numeric(format(acq_date, "%Y")),
    month    = as.numeric(format(acq_date, "%m"))
  ) %>%
  select(latitude, longitude, acq_date, year, month, brightness, confidence, frp, daynight)

# 4. Reshape NASA POWER climate data from wide (jan:dec) to tidy long format
climate_long <- climate_raw %>%
  rename_with(tolower) %>%
  pivot_longer(
    cols      = jan:dec,
    names_to  = "month_name",
    values_to = "value"
  ) %>%
  mutate(
    month = match(month_name, tolower(month.abb)),
    value = as.numeric(value)
  ) %>%
  filter(!is.na(value), value != -999) %>%                                 # Drop NASA POWER missing-value flags (-999)
  distinct(parameter, year, month, lat, lon, .keep_all = TRUE)

# 5. Aggregate both datasets to monthly level and join with left_join()
modis_monthly <- modis_clean %>%
  group_by(year, month) %>%
  summarise(
    fire_count = n(),
    mean_frp   = mean(frp, na.rm = TRUE),
    .groups    = "drop"
  )

climate_monthly <- climate_long %>%
  group_by(year, month, parameter) %>%
  summarise(
    mean_value = mean(value, na.rm = TRUE),
    .groups    = "drop"
  )

nepal_fire_climate_clean <- climate_monthly %>%
  left_join(modis_monthly, by = c("year", "month"))

# 6. Check row counts before and after cleaning, and check join matches
nrow(modis_raw)
nrow(modis_clean)
nrow(climate_raw)
nrow(climate_long)
anti_join(modis_monthly, climate_monthly, by = c("year", "month"))

# 7. Write clean CSVs to clean_data/
write_csv(modis_clean, "clean_data/nepal_modis_clean.csv")
write_csv(nepal_fire_climate_clean, "clean_data/nepal_fire_climate_clean.csv")

# Confirm both clean files are saved in clean_data/
list.files("clean_data")

[1] "raw_data/Nepal_climate_data_2015-2025/Eastern_Nepal_Precpsum.csv"    
[2] "raw_data/Nepal_climate_data_2015-2025/Eastern_Nepal_rlthumat2m.csv"  
[3] "raw_data/Nepal_climate_data_2015-2025/Eastern_Nepal_tempat2m_max.csv"
[4] "raw_data/Nepal_climate_data_2015-2025/Eastern_Nepal_tempat2m.csv"    
[5] "raw_data/Nepal_climate_data_2015-2025/Western_Nepal_precpsum.csv"    
[6] "raw_data/Nepal_climate_data_2015-2025/Western_Nepal_rlthumat2m.csv"  
[7] "raw_data/Nepal_climate_data_2015-2025/Western_Nepal_tempat2m_max.csv"
[8] "raw_data/Nepal_climate_data_2015-2025/Western_Nepal_tempat2m.csv"    


[1] 42501

[1] 39745

[1] 5544

[1] 66528

year,month,fire_count,mean_frp
<dbl>,<dbl>,<int>,<dbl>


[1] "nepal_fire_climate_clean.csv" "nepal_modis_clean.csv"

In [13]:
write_csv(modis_clean, "clean_data/nepal_modis_clean.csv")
write_csv(nepal_fire_climate_clean, "clean_data/nepal_fire_climate_clean.csv")

# Confirm both clean files are saved in clean_data/
list.files("clean_data")

[1] "nepal_fire_climate_clean.csv" "nepal_modis_clean.csv"